# Find stream candidates

Builds the list of the stream candidates to investigate: `data/candidates/candidates_info.csv` (local, gitignored), one row per candidate: `id`, the (ra, dec) of two points along the stream (`endpoint_1_ra`, `endpoint_1_dec`, `endpoint_2_ra`, `endpoint_2_dec`) and the distance modulus where it shows best by eye (`best_dm_by_eye`); `best_dm`, the distance modulus of its largest SNR, is computed in section 7 (`update_candidate_best_dm`). These are the inputs of `CandidateInvestigation` (notebook `candidate_investigation_example`).

The matched-filter map of the whole footprint at one distance modulus (as in `generic_search_example`), with the known streams and the candidates already listed over it; click two points along a new stream on the map; check it in its frame over distance moduli; add it to the list.

Clicking needs the interactive backend of ipympl: `%matplotlib widget` (works in VSCode and Jupyter).

Steps:
1. inputs;
2. stars, footprint and the search, prepared once;
3. the map at one distance modulus, the known streams and the listed candidates; click the endpoints;
4. check in the frame of the endpoints, over distance moduli: the best distance modulus;
5. add the candidate to the list;
6. all the listed candidates on the map.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import logging

import matplotlib.pyplot as plt
import numpy as np

from streameucliddes.analysis.candidate_investigation import CandidateInvestigation, update_candidate_best_dm
from streameucliddes.analysis.cmd_match_filter import get_match_filter
from streameucliddes.analysis.generic_stream_search import generic_search_single_dm, prepare_generic_search
from streameucliddes.catalogs.isochrone_parameters import get_isochrone
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.galstreams_catalog import (
    galstreams_default_tracks,
    load_galstreams,
    select_galstreams_in_footprint,
)
from streameucliddes.catalogs.stream_candidates import DEFAULT_CANDIDATES_PATH, add_candidate, load_candidates
from streameucliddes.catalogs.survey_footprint import survey_footprint_healpix_mask
from streameucliddes.plotting.plot_candidate_investigation import (
    EndpointPicker,
    plot_candidate_frame_check,
    plot_candidate_map,
    plot_candidates,
)
from streameucliddes.plotting.plot_generic_search import plot_generic_search_residual_map

## 1. Inputs

The data and the star sample (as in `generic_search_example`), the match filter and the isochrone (the defaults: `get_match_filter()`, `get_isochrone()`), and the distance modulus of the map to click on (`MAP_DM`). `DISTANCE_MODULI`: the grid of the check in the frame (section 4).

In [ ]:
euclid_release = "dr1"
des = False
data_name = f"euclid_{euclid_release}" + ("_des" if des else "")
subset = "south"  # or "northwest", "northeast", None (all)

star_selection = "morphology_cuts"  # or "xgboost_classifier" (see default_star_cuts)
star_cuts = default_star_cuts(data_name, star_selection)

MATCH_FILTER = get_match_filter()  # the default: Euclid DR1 VIS vs VIS - H, template-fit VIS, its width and red cap
ISOCHRONE = get_isochrone()  # the default: 13 Gyr, [Fe/H] = -1.9 in the default isochrone model
NSIDE = 512  # pixels of about 0.11 deg
SMOOTHING_DEG = 0.3
MAP_DM = 16.5  # distance modulus of the map to click on
DISTANCE_MODULI = np.round(np.arange(15.0, 20.0 + 0.125, 0.25), 2)  # check in the frame (section 4)

## 2. Stars, footprint and the search, prepared once

In [ ]:
stars = get_euclid_stars(data_name, cuts=star_cuts, subset=subset)
footprint = survey_footprint_healpix_mask(stars["ra"].to_numpy(), stars["dec"].to_numpy(), NSIDE, nest=False)
search = prepare_generic_search(stars, match_filter=MATCH_FILTER, nside=NSIDE, footprint_mask=footprint, nest=False)
print(f"{len(stars):,} stars, {search.keep.sum():,} kept")

In [ ]:
# The known streams crossing the footprint (galstreams), one track each
mws = load_galstreams()
selection = select_galstreams_in_footprint(mws, footprint, nest=False)
known_tracks = [
    (row.stream_name, mws[row.track_name].track.ra.deg, mws[row.track_name].track.dec.deg)
    for row in galstreams_default_tracks(selection, mws).itertuples()
]
print(f"{len(known_tracks)} known streams in the footprint")

## 3. The map, and the endpoints by clicking

The corrected matched-filter map at `MAP_DM` (`generic_search_single_dm`), the known streams (dashed) and the candidates already listed (solid, with their id: `plot_candidates`).

Click two points along a new stream: left click adds an endpoint, the second one draws the great-circle arc between them, a third click starts a new pair; right click removes the last endpoint. Zoom or pan with the toolbar if needed: a drag zooms or pans, a click (press and release at the same place) picks an endpoint, with or without the tool on. For another stream, click again on the same map (no need to run the cell again), then run the next cells.

In [ ]:
result = generic_search_single_dm(distance_modulus=MAP_DM, isochrone=ISOCHRONE, search=search,
                                  smoothing_deg=SMOOTHING_DEG)

In [ ]:
fig, ax = plot_generic_search_residual_map(result, tracks=known_tracks, label_tracks=True)
plot_candidates(ax)
picker = EndpointPicker(ax)

In [ ]:
endpoint_1, endpoint_2 = picker.endpoints
print(f"Endpoints: ({endpoint_1[0]:}, {endpoint_1[1]:}) and ({endpoint_2[0]:}, {endpoint_2[1]:}) deg")

## 4. Check in the frame of the endpoints: the best distance modulus

A `CandidateInvestigation` of the clicked endpoints: the frame check (the map with the great circle, and the map in phi1/phi2, at `MAP_DM`), then the maps in the frame over `DISTANCE_MODULI`. Choose the best distance modulus by eye: `BEST_DM_BY_EYE`. `MAP_PHI1_RANGE` and `MAP_PHI2_RANGE` zoom the maps in phi1/phi2 (deg; None: the whole window), and their colour scale is computed on the part shown: set them to leave out a dense object (a dwarf, a cluster, the LMC) that saturates the scale. To move the endpoints, click again in section 3 and run this section again, or edit them here.

In [ ]:

cand = CandidateInvestigation(
    "new candidate", endpoint_1, endpoint_2, age=ISOCHRONE["age"], metallicity=ISOCHRONE["metallicity"],
    match_filter=MATCH_FILTER, distance_moduli=DISTANCE_MODULI,
)
cand.sky_map = result  # the map of section 3: not computed again
cand.prepare(stars, footprint_mask=footprint)  # the stars of the window
cand.compute_zoom_map(MAP_DM)
fig, axes = plot_candidate_frame_check(cand,)

In [ ]:
%%time
cand.scan_distance_moduli()  # n_workers=4 to compute the maps in parallel
print(f"{len(cand.maps)} maps, dm {min(cand.maps):g} to {max(cand.maps):g}")

In [ ]:
MAP_PHI1_RANGE = None  # e.g. (-5.0, 5.0): part of the maps in the frame shown, and its colour scale (deg)
MAP_PHI2_RANGE = None  # e.g. (-2.0, 3.0)
#MAP_PHI2_RANGE = (-2,2)
#MAP_PHI1_RANGE = (-3,2)

In [ ]:
dm_plotted = [15.75,16.,16.25, 16.5, 16.75, 17.0, 17.25, 17.5, 17.75, 18.0]  # distance moduli to plot (the maps are already computed)
#dm_plotted = [17.0, 17.25, 17.5, 17.75, 18.0, 18.25, 18.5]
#for dm in dm_plotted:
#    fig, ax = plot_candidate_map(cand, dm, phi1_range=MAP_PHI1_RANGE, phi2_range=MAP_PHI2_RANGE)

In [ ]:
BEST_DM_BY_EYE = MAP_DM  # by eye, from the maps above
#BEST_DM_BY_EYE = 17.

## 5. Add the candidate to the list

`add_candidate` gives it the next id and writes the file. Run the cell once per candidate: running it again adds the same candidate twice (remove the extra row in the CSV file by hand).

In [ ]:
candidates = add_candidate(endpoint_1, endpoint_2, BEST_DM_BY_EYE)
print(DEFAULT_CANDIDATES_PATH)
candidates

## 6. All the listed candidates on the map

In [ ]:
candidates = load_candidates(include_bad_candidates=True)
fig, ax = plot_generic_search_residual_map(result, tracks=known_tracks)
plot_candidates(ax, candidates)
ax.legend(loc="upper right", fontsize="small", framealpha=0.8)

In [ ]:
import os
fig.savefig(os.path.join("/Users/pelissier/Documents/Codes/packages/euclidDESstreams/figures/outputs/candidates", f"candidates_map.png"), dpi=300)

## 7. Best distance modulus of the listed candidates

`update_candidate_best_dm`: for each candidate without `best_dm`, its SNR against the distance modulus
(`CandidateInvestigation.snr_against_distance_modulus`: the stars in the match filter, on its track against the off
regions on its sides), its regions fitted at its `best_dm_by_eye` (`fit_width`, `define_regions`); `best_dm`: the
distance modulus of the largest SNR, written to the list after each candidate. With the defaults of
`CandidateInvestigation` (the default match filter and isochrone). The candidates already done are skipped:
`REPLACE_FORCE = True` computes them all again. Its progress is logged.

In [ ]:
REPLACE_FORCE = False  # True: compute the best_dm of every candidate again
BEST_DM_GRID = np.round(np.arange(15.5, 19.0 + 0.05, 0.1), 2)  # distance moduli of the SNR

logging.basicConfig(format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
logging.getLogger("streameucliddes").setLevel(logging.INFO)
candidates = update_candidate_best_dm(stars, distance_moduli=BEST_DM_GRID, replace_force=REPLACE_FORCE)
candidates[["id", "best_dm_by_eye", "best_dm"]]